<a href="https://colab.research.google.com/github/AI4Finance-Foundation/FinRL-Tutorials/blob/master/2-Advance/FinRL_Ensemble_StockTrading_ICAIF_2020.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Content

<a id='0'></a>
# Part 1. Problem Definition

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
import datetime

%matplotlib inline
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent,DRLEnsembleAgent
from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline

from pprint import pprint

import sys
sys.path.append("*****/FinRL-Library")

import itertools

C:\Users\perfe\AppData\Roaming\Python\Python39\site-packages\pandas_datareader\compat\__init__.py:11: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  PANDAS_VERSION = LooseVersion(pd.__version__)
C:\Users\perfe\AppData\Roaming\Python\Python39\site-packages\sklearn\utils\multiclass.py:14: DeprecationWarning: Please import `spmatrix` from the `scipy.sparse` namespace; the `scipy.sparse.base` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.sparse.base import spmatrix
C:\Users\perfe\AppData\Roaming\Python\Python39\site-packages\sklearn\utils\optimize.py:18: DeprecationWarning: `scipy.optimize.linesearch.line_search_wolfe2` is deprecated along with the `scipy.optimize.linesearch` namespace. `scipy.optimize.linesearch.line_search_wolfe2` will be removed in SciPy 1.14.0, and the `scipy.optimize.linesearch` namespace will be removed in SciPy 2.0.0.
  from scipy.optimize.linesearch import line_search_wolfe2, line_search_wol

In [3]:
data=pd.read_csv('******.CSV')

In [4]:
data

,tic,date,high,low,close
0,PAD,2018/1/2,13.93,13.32,13.70
1,SPDB,2018/1/2,12.77,12.60,12.72
2,HXB,2018/1/2,9.13,9.01,9.11
3,CMBC,2018/1/2,8.57,8.41,8.52
4,CMB,2018/1/2,30.06,29.02,29.62
...,...,...,...,...,...
25480,CEB,2024/12/31,3.97,3.86,3.87
25481,CZB,2024/12/31,3.02,2.90,2.91
25482,CCB,2024/12/31,8.85,8.75,8.79
25483,BOC,2024/12/31,5.59,5.50,5.51


In [5]:
import os
from finrl.main import check_and_make_directories
from finrl.config import (
    DATA_SAVE_DIR,
    TRAINED_MODEL_DIR,
    TENSORBOARD_LOG_DIR,
    RESULTS_DIR,
    INDICATORS,
)
check_and_make_directories([DATA_SAVE_DIR, TRAINED_MODEL_DIR, TENSORBOARD_LOG_DIR, RESULTS_DIR])


In [6]:
data.sort_values(['date']).head()

,tic,date,high,low,close
90,PAD,2018/1/10,13.49,12.92,13.470000
91,SPDB,2018/1/10,13.09,12.68,13.020000
92,HXB,2018/1/10,9.37,9.17,9.350000
93,CMBC,2018/1/10,8.84,8.63,8.820000
94,CMB,2018/1/10,30.77,29.58,30.529999


In [7]:
data.sort_values(['date']).tail()

,tic,date,high,low,close
24362,HXB,2024/9/9,6.59,6.44,6.52
24361,SPDB,2024/9/9,8.47,8.32,8.38
24360,PAD,2024/9/9,10.07,9.84,9.85
24373,BOC,2024/9/9,4.74,4.62,4.68
24374,ECITIC,2024/9/9,5.90,5.78,5.82


In [8]:
len(data.tic.unique())

15

In [9]:
data.tic.value_counts()

tic
PAD       1699
SPDB      1699
HXB       1699
CMBC      1699
CMB       1699
CIB       1699
ABC       1699
BOCOM     1699
ICBC      1699
PSBC      1699
CEB       1699
CZB       1699
CCB       1699
BOC       1699
ECITIC    1699
Name: count, dtype: int64

In [10]:
fe = FeatureEngineer(use_technical_indicator=True,
                     tech_indicator_list = INDICATORS,
                     use_turbulence=True,
                     user_defined_feature = False)

processed = fe.preprocess_data(data)
processed = processed.copy()
processed = processed.fillna(0)
processed = processed.replace(np.inf,0)

Successfully added technical indicators
Successfully added turbulence index


In [11]:
processed.sample(5)

,tic,date,high,low,close,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,turbulence
14264,SPDB,2021/8/5,9.120,9.040000,9.050,-0.121835,10.288065,8.531435,46.126720,-71.368297,2.147104,9.502167,9.785083,2.665194
18149,SPDB,2022/9/30,7.070,6.990000,7.040,-0.121027,8.649619,5.941881,48.297410,-58.229337,0.795700,7.354333,7.824833,3.098960
18206,ICBC,2022/9/8,4.360,4.340000,4.350,-0.003175,5.466259,3.563741,48.768410,-39.180827,1.316904,4.512833,4.670750,1.322725
7795,HXB,2020/11/10,7.635,7.395001,7.595,-0.102897,7.829959,5.343041,56.025161,61.133046,30.250213,6.921333,7.206583,0.407248
1719,ECITIC,2018/3/5,6.760,6.610000,6.680,-0.142498,7.801226,5.830774,49.750037,-37.983293,6.005848,7.075667,6.723500,0.000000


In [12]:
stock_dimension = len(processed.tic.unique())
state_space = 1 + 2*stock_dimension + len(INDICATORS)*stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")


Stock Dimension: 15, State Space: 151


In [13]:
env_kwargs = {
    "hmax": 100, 
    "initial_amount": 1000000, 
    "buy_cost_pct": 0.001, 
    "sell_cost_pct": 0.001, 
    "state_space": state_space, 
    "stock_dim": stock_dimension, 
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension, 
    "reward_scaling": 1e-4,
    "print_verbosity":5
    
}

In [14]:

TRAIN_START_DATE = '2018-01-02'
TRAIN_END_DATE = '2022-12-30'
TEST_START_DATE = '2023-01-03'
TEST_END_DATE = '2024-12-31'

In [15]:
rebalance_window = 63 # rebalance_window is the number of days to retrain the model
validation_window = 63 # validation_window is the number of days to do validation and trading (e.g. if validation_window=63, then both validation and trading period will be 63 days)

ensemble_agent = DRLEnsembleAgent(df=processed,
                 train_period=(TRAIN_START_DATE,TRAIN_END_DATE),
                 val_test_period=(TEST_START_DATE,TEST_END_DATE),
                 rebalance_window=rebalance_window, 
                 validation_window=validation_window, 
                 **env_kwargs)



In [16]:
A2C_model_kwargs = {
                    'n_steps': 10,
                    'ent_coef': 0.005,
                    'learning_rate': 0.00047
                    }

PPO_model_kwargs = {
                    "ent_coef":0.005,
                    "n_steps": 2048,
                    "learning_rate": 0.001,
                    "batch_size": 128
                    }

DDPG_model_kwargs = {
                      #"action_noise":"ornstein_uhlenbeck",
                      "buffer_size": 50_000,
                      "learning_rate": 0.001,
                      "batch_size": 128
                    }
TD3_model_kwargs = {
                      #"action_noise":"ornstein_uhlenbeck",
                      "buffer_size": 100_000,
                      "learning_rate": 0.001,
                      "batch_size": 100
                    }
SAC_model_kwargs = {
                      #"action_noise":"ornstein_uhlenbeck",
                      "buffer_size": 100_000,
                      "learning_rate": 0.0001,
                      "batch_size": 128
                    }
timesteps_dict = {'a2c' : 20_000, 
                 'ppo' : 20_000, 
                 'ddpg' : 20_000,
                  'sac':20_000,
                 'td3' : 20_000,
                 }

In [18]:
df_summary = ensemble_agent.run_ensemble_strategy(A2C_model_kwargs,
                                                 PPO_model_kwargs,
                                                 DDPG_model_kwargs,
                                                 TD3_model_kwargs,
                                                  SAC_model_kwargs,
                                                 timesteps_dict)

============Start Ensemble Strategy============
turbulence_threshold:  217.5062371450193
======Model training from:  2018-01-02 to  2023/1/10
======a2c Training========
{'n_steps': 10, 'ent_coef': 0.005, 'learning_rate': 0.00047}
Using cuda device
Logging to tensorboard_log/a2c\a2c_126_75
-------------------------------------
| time/                 |           |
|    fps                | 152       |
|    iterations         | 100       |
|    time_elapsed       | 6         |
|    total_timesteps    | 1000      |
| train/                |           |
|    entropy_loss       | -21.3     |
|    explained_variance | 0.0145    |
|    learning_rate      | 0.00047   |
|    n_updates          | 99        |
|    policy_loss        | 75.4      |
|    reward             | 33.568745 |
|    std                | 0.999     |
|    value_loss         | 49        |
-------------------------------------
-------------------------------------
| time/                 |           |
|    fps                | 

ValueError: Expected parameter loc (Tensor of shape (1, 15)) of distribution Normal(loc: torch.Size([1, 15]), scale: torch.Size([1, 15])) to satisfy the constraint Real(), but found invalid values:
tensor([[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]],
       device='cuda:0')

ValueError: If using all scalar values, you must pass an index

In [ ]:
df_summary

In [ ]:
unique_trade_date = processed[(processed.date > TEST_START_DATE)&(processed.date <= TEST_END_DATE)].date.unique()

In [ ]:
df_trade_date = pd.DataFrame({'datadate':unique_trade_date})

df_account_value=pd.DataFrame()
for i in range(rebalance_window+validation_window, len(unique_trade_date)+1,rebalance_window):
    temp = pd.read_csv('results/account_value_trade_{}_{}.csv'.format('ensemble',i))
    df_account_value =  pd.concat([df_account_value, temp], ignore_index=True)
sharpe=(252**0.5)*df_account_value.account_value.pct_change(1).mean()/df_account_value.account_value.pct_change(1).std()
print('Sharpe Ratio: ',sharpe)
df_account_value=df_account_value.join(df_trade_date[validation_window:].reset_index(drop=True))

In [ ]:
df_account_value.head()

In [22]:
df_account_value.to_csv('******.csv')

In [ ]:
%matplotlib inline
df_account_value.account_value.plot()

<a id='6.1'></a>
## 7.1 BackTestStats
pass in df_account_value, this information is stored in env class


In [ ]:
print("==============Get Backtest Results===========")
now = datetime.datetime.now().strftime('%Y%m%d-%Hh%M')

perf_stats_all = backtest_stats(account_value=df_account_value)
perf_stats_all = pd.DataFrame(perf_stats_all)

In [25]:
df_account_value

,account_value,date,daily_return,datadate
0,1000000.000,2023/12/20,NaN,2023/12/20
1,1000010.224,2023/12/21,0.000010,2023/12/21
2,1000040.434,2023/12/22,0.000030,2023/12/22
3,1000045.627,2023/12/25,0.000005,2023/12/25
4,1000005.817,2023/12/26,-0.000040,2023/12/26
...,...,...,...,...
121,1009205.464,2023/7/18,-0.002141,2023/7/18
122,1010258.226,2023/7/19,0.001043,2023/7/19
123,1008419.965,2023/7/20,-0.001820,2023/7/20
124,1008846.743,2023/7/21,0.000423,2023/7/21
